# [6.4] Crosscoders and Model Diffing - Solutions

Reference validation notebook for the section-local crosscoder implementation. This executes the visible tests against `solutions.py`, checks the CPU notebook contract, and verifies the committed CUDA report highlights.

Expected CUDA highlights: pinned `gelu-1l` and `solu-1l` load on CUDA, exact shared-plus-delta reconstruction passes, the top SVD model-diff direction separates generated technical vs everyday prompt labels, and top-direction ablation reduces activation-space model deltas much more than an orthogonal random direction.


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter6_sparse_feature_methods"
section = "part4_crosscoders_model_diffing"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_crosscoders_model_diffing.tests as tests
from part4_crosscoders_model_diffing import solutions


In [ ]:
tests.test_decode_crosscoder_reconstructs_shared_and_specific_spaces(
    solutions.decode_crosscoder,
    solutions.crosscoder_reconstruction_report,
)
tests.test_feature_specificity_classifies_shared_model_a_and_model_b(
    solutions.feature_specificity_report,
    solutions.classify_features_by_specificity,
)
tests.test_behavior_delta_prediction_uses_signed_auc_and_means(
    solutions.behavior_delta_prediction_report,
    solutions.roc_auc_binary,
)
tests.test_toy_behavior_delta_scores_are_model_b_minus_model_a(
    solutions.toy_behavior_delta_scores,
)
tests.test_crosscoder_ablation_requires_target_to_beat_random_control(
    solutions.crosscoder_ablation_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
contract


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"], "6.4 CUDA preflight should pass."
assert gpu["model_a_name"] == "gelu-1l", "Model A should remain pinned to gelu-1l."
assert gpu["model_b_name"] == "solu-1l", "Model B should remain pinned to solu-1l."
assert gpu["model_a_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603", "gelu-1l revision should remain pinned."
assert gpu["model_b_revision"] == "a4ce32db5e35f13e5f09333888bd2d42660f77ce", "solu-1l revision should remain pinned."
assert gpu["shared_reconstructs_both"], "Exact shared-plus-delta reconstruction should pass both model spaces."
assert gpu["behavior_delta_auc"] == 1.0, "Top model-diff direction should separate technical vs everyday prompts."
assert gpu["top_variance_fraction"] >= 0.25, "Top SVD direction should explain a nontrivial share of cross-model variance."
assert gpu["delta_reduction"] >= 1.0, "Top-direction ablation should strongly reduce activation-space model deltas."
assert gpu["random_reduction"] <= 0.1, "Orthogonal random-direction ablation should be a weak control."
assert gpu["ablation_passes_control"], "Top-direction ablation should beat the random-direction control."
assert gpu["floor_top_delta_abs_mean"] >= 0.1, "Behavioral logit readout should be nontrivial."
assert gpu["peak_vram_gb"] <= 1.0, "The paired-model preflight should stay under the locked 1GB budget."
{
    "behavior_delta_auc": gpu["behavior_delta_auc"],
    "top_variance_fraction": gpu["top_variance_fraction"],
    "delta_reduction": gpu["delta_reduction"],
    "random_reduction": gpu["random_reduction"],
    "floor_top_delta_abs_mean": gpu["floor_top_delta_abs_mean"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
